# Import libraries

In [ ]:
import psycopg2
from pgvector.psycopg2 import register_vector

import numpy as np 
from sklearn.decomposition import PCA
import time
import umap
# import pacmap

# Get high-dim data

In [ ]:
def get_high_dim_data_ML(table_name, EM_name, ML_val, additional_info): 

    conn = psycopg2.connect(
        dbname  = "embeddings_db",
        user    = "",
        host    = "localhost",
        port    = "5432"
    )

    register_vector(conn); 
    cur = conn.cursor()
    # cur.execute(f"SELECT embedding_data FROM {table_name} WHERE embedding_max_length = {ML_val}")  

    cur.execute(f"""
        SELECT embedding_data
        FROM {table_name}
        WHERE embedding_model = %s AND embedding_max_length = %s AND additional_info_inc = %s
    """, (EM_name, ML_val, additional_info))  


    embedding_arr = np.array([row[0] for row in cur.fetchall()])
    return embedding_arr

# Low-dim data - PCA & UMAP

In [ ]:
def apply_pca_to_data(input_data, num_dim):

    pca                 = PCA(n_components=num_dim, random_state=42)
    reduced_data        = pca.fit_transform(input_data)
    reduced_data        = np.array(reduced_data)

    return reduced_data

# UMAP - set to a two dimensional space 
def apply_umap_to_data(input_data, num_dim, num_neigh): 

    np.random.seed(42)
     # want to visualise the data so pick 2 dimensions
    num_components = num_dim
    num_neighbours = num_neigh

    # create the umap model we are going to be applying to the fitting
    umap_model = umap.UMAP(n_neighbors=num_neighbours, n_components=num_components, random_state=42)
    umap_fit = umap_model.fit_transform(input_data)
    umap_fit   = np.array(umap_fit)

    return umap_fit

# PaCMAP
def apply_pacmap_to_data(input_data, num_dim, num_neigh, MN_ratio_val, FP_ratio_val): 

    np.random.seed(42)
    # create and fit the pacmap model 
    pacmap_model    = pacmap.PaCMAP(n_components=num_dim, n_neighbors=num_neigh, MN_ratio=MN_ratio_val, FP_ratio=FP_ratio_val, random_state=42)
    pacmap_fit      = pacmap_model.fit_transform(input_data)
    pacmap_fit      = np.array(pacmap_fit)

    return pacmap_fit

In [ ]:
additional_info = "organization_newsgroup_body"
dim_arr         = [5, 10, 50]

## Functions

In [ ]:
def add_low_dim_pca_data_20NG(low_dim_embedding, additional_info, EM_name, EM_ML, time, dim_size): 

    # include variables
    dataset_info        = '20NG'
    dr_method_info      = 'PCA'
    dr_parameter_val    = 'N/A'
    MN_ratio_val        = 'N/A'
    FP_ratio_val        = 'N/A'

    # Connect to PostgreSQL
    conn = psycopg2.connect(
        dbname  = "embeddings_db",
        user    = "",
        host    = "localhost",
        port    = "5432"
    )

    # 20NG PCA Data
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO LOW_DIM_EMBEDDINGS_20NG (  dataset,        additional_info,    embedding_model,    embedding_max_length,   dr_method,      dr_parameter,       mn_ratio,       fp_ratio,       dimension_size, low_dim_embedding,          time_seconds) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)",
        (                                       dataset_info,   additional_info,    EM_name,            EM_ML,                  dr_method_info, dr_parameter_val,   MN_ratio_val,   FP_ratio_val,   dim_size,       low_dim_embedding.tolist(), time)
    )

    conn.commit()
    cursor.close()
    conn.close()

In [ ]:
def calc_PCA_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label):

    for i in range(len(dim_arr)):
        start_time      = time.time()
        low_dim_data    = apply_pca_to_data(high_dim_data, dim_arr[i])
        end_time        = time.time()
        duration        = end_time - start_time

        print(low_dim_data.shape)

        add_low_dim_pca_data_20NG(low_dim_data, additional_info, EM_arr_label, L_arr_label, duration, dim_arr[i])

In [ ]:
def add_low_dim_umap_data_20NG(low_dim_embedding, additional_info, EM_name, EM_ML, time, dim_size, num_neigh): 

    # include variables
    dataset_info        = '20NG'
    dr_method_info      = 'UMAP'
    dr_parameter_val    = str(num_neigh)
    MN_ratio_val        = 'N/A'
    FP_ratio_val        = 'N/A'

    # Connect to PostgreSQL
    conn = psycopg2.connect(
        dbname  = "embeddings_db",
        user    = "",
        host    = "localhost",
        port    = "5432"
    )

    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO LOW_DIM_EMBEDDINGS_20NG (  dataset,        additional_info,    embedding_model,    embedding_max_length,   dr_method,      dr_parameter,       mn_ratio,       fp_ratio,       dimension_size, low_dim_embedding,          time_seconds) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)",
        (                                       dataset_info,   additional_info,    EM_name,            EM_ML,                  dr_method_info, dr_parameter_val,   MN_ratio_val,   FP_ratio_val,   dim_size,       low_dim_embedding.tolist(), time)
    )

    conn.commit()
    cursor.close()
    conn.close()

def calc_UMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_name, L_val):

    num_neigh_val   = 30

    for i in range(len(dim_arr)): 

        start_time      = time.time()
        low_dim_data    = apply_umap_to_data(high_dim_data, dim_arr[i], num_neigh_val)
        end_time        = time.time()
        duration        = end_time - start_time

        print(low_dim_data.shape)

        add_low_dim_umap_data_20NG(low_dim_data, additional_info, EM_name, L_val, duration, dim_arr[i], num_neigh_val)

## BERT-128

In [ ]:
EM_arr_label    = "google-bert/bert-base-uncased"
EM_short_name   = "bert"
L_arr_label     = 128
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
# PCA
calc_PCA_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

In [ ]:
# UMAP
calc_UMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## BERT-256

In [ ]:
EM_arr_label    = "google-bert/bert-base-uncased"
EM_short_name   = "bert"
L_arr_label     = 256
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
# PCA
calc_PCA_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

In [ ]:
# UMAP
calc_UMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## BERT-512

In [ ]:
EM_arr_label    = "google-bert/bert-base-uncased"
EM_short_name   = "bert"
L_arr_label     = 512
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
# PCA
calc_PCA_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

In [ ]:
# UMAP
calc_UMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## All-MiniLM 128

In [ ]:
EM_arr_label    = "sentence-transformers/all-MiniLM-L6-v2"
EM_short_name   = "allminilm"
L_arr_label     = 128
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
# PCA
calc_PCA_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

In [ ]:
# UMAP
calc_UMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## All-MiniLM 256

In [ ]:
EM_arr_label    = "sentence-transformers/all-MiniLM-L6-v2"
EM_short_name   = "allminilm"
L_arr_label     = 256
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
# PCA
calc_PCA_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

In [ ]:
# UMAP
calc_UMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## All-MiniLM 512

In [ ]:
EM_arr_label    = "sentence-transformers/all-MiniLM-L6-v2"
EM_short_name   = "allminilm"
L_arr_label     = 512
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
# PCA
calc_PCA_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

In [ ]:
# UMAP
calc_UMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## E5 128

In [ ]:
EM_arr_label    = "intfloat/e5-small-v2"
EM_short_name   = "e5"
L_arr_label     = 128
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
# PCA
calc_PCA_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

In [ ]:
# UMAP
calc_UMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## E5 256

In [ ]:
EM_arr_label    = "intfloat/e5-small-v2"
EM_short_name   = "e5"
L_arr_label     = 256
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
# PCA
calc_PCA_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

In [ ]:
# UMAP
calc_UMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## E5 512

In [ ]:
EM_arr_label    = "intfloat/e5-small-v2"
EM_short_name   = "e5"
L_arr_label     = 512
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
# PCA
calc_PCA_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

In [ ]:
# UMAP
calc_UMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## Longformer 1024

In [ ]:
EM_arr_label    = "allenai/longformer-base-4096"
EM_short_name   = "longformer"
L_arr_label     = 1024
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)
# print(high_dim_data[18])

In [ ]:
# PCA
calc_PCA_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

In [ ]:
# UMAP
calc_UMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## Longformer 2048

In [ ]:
EM_arr_label    = "allenai/longformer-base-4096"
EM_short_name   = "longformer"
L_arr_label     = 2048
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)
# print(high_dim_data[18])

In [ ]:
# PCA
calc_PCA_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

In [ ]:
# UMAP
calc_UMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## Longformer 4096

In [ ]:
EM_arr_label    = "allenai/longformer-base-4096"
EM_short_name   = "longformer"
L_arr_label     = 4096
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)
# print(high_dim_data[18])

In [ ]:
# PCA
calc_PCA_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

In [ ]:
# UMAP
calc_UMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## BigBird 1024

In [ ]:
EM_arr_label    = "google/bigbird-roberta-base"
EM_short_name   = "bigbird"
L_arr_label     = 1024
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)
# print(high_dim_data[11])

In [ ]:
# PCA
calc_PCA_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

In [ ]:
# UMAP
calc_UMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## BigBird 2048

In [ ]:
EM_arr_label    = "google/bigbird-roberta-base"
EM_short_name   = "bigbird"
L_arr_label     = 2048
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)
# print(high_dim_data[11])

In [ ]:
# PCA
calc_PCA_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

In [ ]:
# UMAP
calc_UMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## BigBird 4096

In [ ]:
EM_arr_label    = "google/bigbird-roberta-base"
EM_short_name   = "bigbird"
L_arr_label     = 4096
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)
# print(high_dim_data[11])

In [ ]:
# PCA
calc_PCA_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

In [ ]:
# UMAP
calc_UMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## Qwen3 1024

In [ ]:
EM_arr_label    = "qwen3-embedding:8b"
EM_short_name   = "qwen3"
L_arr_label     = 1024
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)
# print(high_dim_data[11])

In [ ]:
# PCA
calc_PCA_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

In [ ]:
# UMAP
calc_UMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## Qwen3 2048

In [ ]:
EM_arr_label    = "qwen3-embedding:8b"
EM_short_name   = "qwen3"
L_arr_label     = 2048
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)
# print(high_dim_data[11])

In [ ]:
# PCA
calc_PCA_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

In [ ]:
# UMAP
calc_UMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## Qwen3 4096

In [ ]:
EM_arr_label    = "qwen3-embedding:8b"
EM_short_name   = "qwen3"
L_arr_label     = 4096
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = get_high_dim_data_ML(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)
# print(high_dim_data[11])

In [ ]:
# PCA
calc_PCA_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

In [ ]:
# UMAP
calc_UMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

# PaCMAP

In [ ]:
# run pacmap_env (Python 3.11.14) in the code for PaCMAP to work
# for PaCMAP to run, it needs to run on x86_64 not arm64, so it is working with Rosetta now

import pacmap, numpy as np, pandas as pd
import time

from sqlalchemy import create_engine, MetaData, Table, insert, text

In [ ]:
# function to write to postico, using sqlalchemy 

def parse_vector(vector_str): 
    return np.fromstring(vector_str.strip("[]"), sep=",")

def read_sql_alchemy_high_dim_data(table_name, EM_name, ML_val, additional_info): 

    engine = create_engine("postgresql+pg8000://@localhost:5432/embeddings_db")

    with engine.connect() as conn: 
        result = conn.execute(
            text(f"SELECT embedding_data FROM {table_name} WHERE embedding_model = :em AND embedding_max_length = :ml AND additional_info_inc = :add_info"), 
            {"em": EM_name, "ml": ML_val, "add_info": additional_info}
        )
        embedding_arr = np.array([parse_vector(row[0]) for row in result])
    return embedding_arr

def apply_pacmap_to_data(input_data, num_dim, num_neigh, MN_ratio_val, FP_ratio_val): 

    # create and fit the pacmap model 
    pacmap_model    = pacmap.PaCMAP(n_components=num_dim, n_neighbors=num_neigh, MN_ratio=MN_ratio_val, FP_ratio=FP_ratio_val, random_state=42)
    pacmap_fit      = pacmap_model.fit_transform(input_data)
    pacmap_fit      = np.array(pacmap_fit)

    return pacmap_fit

In [ ]:
# function to write to postico, using sqlalchemy 
def write_20NG_pacmap_embeddings_to_postico(additional_info, em_name, L_val, nn_val, mn_val, fp_val, dim_size, low_dim_data, time): 

    engine          = create_engine("postgresql+pg8000://@localhost:5432/embeddings_db")
    metadata        = MetaData()
    db_table        = Table("low_dim_embeddings_20ng", metadata, autoload_with=engine) 

    row_to_insert = {

        "dataset":                  "20NG", 
        "additional_info":          additional_info,
        "embedding_model":          em_name,
        "embedding_max_length":     L_val,
        "dr_method":                "PaCMAP",
        "dr_parameter":             nn_val, 
        "mn_ratio":                 mn_val, 
        "fp_ratio":                 fp_val, 
        "dimension_size":           dim_size, 
        "low_dim_embedding":        low_dim_data, 
        "time_seconds":             time                      
    }

    with engine.connect() as conn: 
        data = insert(db_table).values(**row_to_insert)
        conn.execute(data)
        conn.commit()

In [ ]:
def calc_PaCMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label):

    dim_arr         = [5, 10, 50]
    num_neigh_arr   = [30]
    MN_ratio_arr    = [2]
    FP_ratio_arr    = [0.5]

    for i in range(len(dim_arr)): 
        for j in range(len(num_neigh_arr)): 
            for k in range(len(MN_ratio_arr)): 
                for l in range(len(FP_ratio_arr)): 

                    start_time      = time.time()
                    low_dim_data    = apply_pacmap_to_data(high_dim_data, dim_arr[i], num_neigh_arr[j], MN_ratio_arr[k], FP_ratio_arr[l])
                    end_time        = time.time()
                    duration        = end_time - start_time
                    write_20NG_pacmap_embeddings_to_postico(additional_info, EM_arr_label, L_arr_label, str(num_neigh_arr[j]), str(MN_ratio_arr[k]), str(FP_ratio_arr[l]), dim_arr[i], low_dim_data.tolist(), duration)
                    print(low_dim_data.shape)

## BERT-128

In [ ]:
EM_arr_label    = "google-bert/bert-base-uncased"
EM_short_name   = "bert"
L_arr_label     = 128
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = read_sql_alchemy_high_dim_data(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
calc_PaCMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## BERT-256

In [ ]:
EM_arr_label    = "google-bert/bert-base-uncased"
EM_short_name   = "bert"
L_arr_label     = 256
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = read_sql_alchemy_high_dim_data(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
calc_PaCMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## BERT-512

In [ ]:
EM_arr_label    = "google-bert/bert-base-uncased"
EM_short_name   = "bert"
L_arr_label     = 512
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = read_sql_alchemy_high_dim_data(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
calc_PaCMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## All-MiniLM-128

In [ ]:
EM_arr_label    = "sentence-transformers/all-MiniLM-L6-v2"
EM_short_name   = "allminilm"
L_arr_label     = 128
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = read_sql_alchemy_high_dim_data(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
calc_PaCMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## All-MiniLM-256

In [ ]:
EM_arr_label    = "sentence-transformers/all-MiniLM-L6-v2"
EM_short_name   = "allminilm"
L_arr_label     = 256
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = read_sql_alchemy_high_dim_data(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
calc_PaCMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## All-MiniLM-512

In [ ]:
EM_arr_label    = "sentence-transformers/all-MiniLM-L6-v2"
EM_short_name   = "allminilm"
L_arr_label     = 512
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = read_sql_alchemy_high_dim_data(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
calc_PaCMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## E5-128

In [ ]:
EM_arr_label    = "intfloat/e5-small-v2"
EM_short_name   = "e5"
L_arr_label     = 128
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = read_sql_alchemy_high_dim_data(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
calc_PaCMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## E5-256

In [ ]:
EM_arr_label    = "intfloat/e5-small-v2"
EM_short_name   = "e5"
L_arr_label     = 256
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = read_sql_alchemy_high_dim_data(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
calc_PaCMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## E5-512

In [ ]:
EM_arr_label    = "intfloat/e5-small-v2"
EM_short_name   = "e5"
L_arr_label     = 512
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = read_sql_alchemy_high_dim_data(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
calc_PaCMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## Longformer-1024

In [ ]:
EM_arr_label    = "allenai/longformer-base-4096"
EM_short_name   = "longformer"
L_arr_label     = 1024
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = read_sql_alchemy_high_dim_data(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
calc_PaCMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## Longformer-2048

In [ ]:
EM_arr_label    = "allenai/longformer-base-4096"
EM_short_name   = "longformer"
L_arr_label     = 2048
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = read_sql_alchemy_high_dim_data(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
calc_PaCMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## Longformer-4096

In [ ]:
EM_arr_label    = "allenai/longformer-base-4096"
EM_short_name   = "longformer"
L_arr_label     = 4096
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = read_sql_alchemy_high_dim_data(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
calc_PaCMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## BigBird-1024

In [ ]:
EM_arr_label    = "google/bigbird-roberta-base"
EM_short_name   = "bigbird"
L_arr_label     = 1024
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = read_sql_alchemy_high_dim_data(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
calc_PaCMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## BigBird-2048

In [ ]:
EM_arr_label    = "google/bigbird-roberta-base"
EM_short_name   = "bigbird"
L_arr_label     = 2048
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = read_sql_alchemy_high_dim_data(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
calc_PaCMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## BigBird-4096

In [ ]:
EM_arr_label    = "google/bigbird-roberta-base"
EM_short_name   = "bigbird"
L_arr_label     = 4096
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = read_sql_alchemy_high_dim_data(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
calc_PaCMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## Qwen3-1024

In [ ]:
EM_arr_label    = "qwen3-embedding:8b"
EM_short_name   = "qwen3"
L_arr_label     = 1024
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = read_sql_alchemy_high_dim_data(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
calc_PaCMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## Qwen3-2048

In [ ]:
EM_arr_label    = "qwen3-embedding:8b"
EM_short_name   = "qwen3"
L_arr_label     = 2048
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = read_sql_alchemy_high_dim_data(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
calc_PaCMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)

## Qwen3-4096

In [ ]:
EM_arr_label    = "qwen3-embedding:8b"
EM_short_name   = "qwen3"
L_arr_label     = 4096
table_name      = f"{EM_short_name}_{additional_info}_20ng.{EM_short_name}_{additional_info}_ml{L_arr_label}_20ng"
print(table_name)

high_dim_data   = read_sql_alchemy_high_dim_data(table_name, EM_arr_label, L_arr_label, additional_info)
print(high_dim_data.shape)

In [ ]:
calc_PaCMAP_low_dim_embeddings_20NG(high_dim_data, additional_info, dim_arr, EM_arr_label, L_arr_label)